# 05b Working Capital Impact - Standardized Input

## Purpose

This notebook is the standardized-input version of `05_working_capital_impact.ipynb`. It reproduces the existing working-capital impact logic using the standardized SKU profile created by `03b_sku_classification_standardized_input.ipynb` and used by the 04b preparation workflow.

This is a preparation layer for future reusable pipeline refactoring. It does not replace the original workflow or change the validated project metrics.

## Input, output, and SKU scope

The notebook consumes:

```text
outputs_standardized/sku_profile_classification.csv
```

It writes five working-capital outputs only to `outputs_standardized/`. The original `outputs/` and `data/processed/` directories are not modified.

`stock_code` remains the normalized SKU key. `description` is retained only as a display field and is not used as a grouping key.

## Important simulation notice

The inventory and cost fields used in this analysis are simulated portfolio assumptions. They are not real company inventory balances, supplier costs, warehouse data, accounting values, or financial exposures.

The resulting metrics are illustrative decision-support measures only. They must not be interpreted as audited accounting amounts or operational recommendations.

## Load the standardized SKU profile

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve paths whether the notebook runs from the project root or notebooks/.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
outputs_dir = project_root / "outputs_standardized"
sku_profile_path = outputs_dir / "sku_profile_classification.csv"

if not sku_profile_path.exists():
    raise FileNotFoundError(
        "Standardized SKU profile not found. Run "
        "notebooks/03b_sku_classification_standardized_input.ipynb first. "
        f"Expected file: {sku_profile_path}"
    )

sku_profile = pd.read_csv(
    sku_profile_path,
    dtype={"stock_code": "string", "description": "string"},
)

required_columns = {
    "stock_code", "description", "sku_class", "avg_monthly_units",
    "avg_unit_price", "demand_cv",
}
missing_columns = sorted(required_columns - set(sku_profile.columns))
if missing_columns:
    raise ValueError(
        "Missing standardized SKU profile columns: " + ", ".join(missing_columns)
    )
if sku_profile["stock_code"].duplicated().any():
    raise ValueError("SKU profile must contain one row per stock_code.")

outputs_dir.mkdir(parents=True, exist_ok=True)

print("SKU profile shape:", sku_profile.shape)
print("Source file:", sku_profile_path)
print("Unique stock_code values:", sku_profile["stock_code"].nunique())
sku_profile.head()

## Use or recreate simulated inventory and cost fields

If the complete simulated inventory layer is already present, the notebook uses it. The standardized field `supplier_lead_time_days` is preferred; an existing original `lead_time_days` field is accepted as a compatibility alias.

If any required simulated field is missing, the notebook recreates the full layer using notebook 05's seed, distributions, calculations, thresholds, and rule precedence.

In [ ]:
# Accept the original lead-time name when a prepared input already contains it.
if (
    "supplier_lead_time_days" not in sku_profile.columns
    and "lead_time_days" in sku_profile.columns
):
    sku_profile["supplier_lead_time_days"] = sku_profile["lead_time_days"]

required_simulated_fields = [
    "current_inventory",
    "supplier_lead_time_days",
    "storage_volume_per_unit",
    "unit_cost",
    "avg_daily_demand",
    "safety_stock",
    "reorder_point",
    "recommended_replenishment_qty",
    "inventory_coverage_days",
    "inventory_risk",
    "warehouse_strategy",
]

missing_fields = [
    field for field in required_simulated_fields if field not in sku_profile.columns
]

if missing_fields:
    print("Missing simulated fields detected:", missing_fields)
    print("Recreating simulated inventory fields using deterministic portfolio assumptions.")

    np.random.seed(42)

    sku_profile["current_inventory"] = np.where(
        sku_profile["sku_class"].isin(["High-Revenue Priority", "High-Turnover Stable"]),
        np.random.randint(20, 300, size=len(sku_profile)),
        np.random.randint(0, 120, size=len(sku_profile)),
    )

    sku_profile["supplier_lead_time_days"] = np.random.choice(
        [7, 14, 21, 30, 45],
        size=len(sku_profile),
        p=[0.20, 0.35, 0.25, 0.15, 0.05],
    )

    sku_profile["storage_volume_per_unit"] = np.random.uniform(
        0.1,
        3.0,
        size=len(sku_profile),
    ).round(2)

    sku_profile["unit_cost"] = (
        sku_profile["avg_unit_price"]
        * np.random.uniform(0.35, 0.65, size=len(sku_profile))
    ).round(2)

    sku_profile["avg_daily_demand"] = sku_profile["avg_monthly_units"] / 30

    sku_profile["safety_stock"] = (
        sku_profile["avg_daily_demand"]
        * sku_profile["supplier_lead_time_days"]
        * (0.25 + sku_profile["demand_cv"].clip(0, 2) * 0.25)
    ).round(0)

    sku_profile["reorder_point"] = (
        sku_profile["avg_daily_demand"] * sku_profile["supplier_lead_time_days"]
        + sku_profile["safety_stock"]
    ).round(0)

    sku_profile["recommended_replenishment_qty"] = (
        sku_profile["reorder_point"] - sku_profile["current_inventory"]
    ).clip(lower=0).round(0)

    sku_profile["inventory_coverage_days"] = np.where(
        sku_profile["avg_daily_demand"] > 0,
        sku_profile["current_inventory"] / sku_profile["avg_daily_demand"],
        np.inf,
    )

    def assign_inventory_risk(row):
        if row["current_inventory"] < row["reorder_point"]:
            return "Stockout Risk"
        if (
            row["inventory_coverage_days"] > 180
            and row["sku_class"] in ["Long-Tail", "Regular"]
        ):
            return "Overstock Risk"
        return "Normal"

    sku_profile["inventory_risk"] = sku_profile.apply(
        assign_inventory_risk,
        axis=1,
    )

    def assign_warehouse_strategy(row):
        if row["inventory_risk"] == "Overstock Risk":
            return "Overstock Review / Reduce Replenishment"
        if row["sku_class"] == "High-Revenue Priority":
            return "Local Warehouse Priority"
        if row["sku_class"] == "High-Turnover Stable":
            return "Stable Local Warehouse Inventory"
        if row["sku_class"] == "High-Turnover Volatile":
            return "Small-Batch Replenishment / Monitor Closely"
        if row["sku_class"] == "Long-Tail":
            return "External or Limited Stock Strategy"
        return "Standard Replenishment Review"

    sku_profile["warehouse_strategy"] = sku_profile.apply(
        assign_warehouse_strategy,
        axis=1,
    )
else:
    print("Using simulated inventory fields already present in the source file.")

sku_profile[
    [
        "stock_code", "description", "sku_class", "current_inventory",
        "unit_cost", "inventory_risk", "warehouse_strategy",
    ]
].head()

## Working-capital impact metrics

The finance-facing calculations match notebook 05:

- `estimated_inventory_value`: simulated current inventory multiplied by simulated unit cost.
- `stockout_revenue_exposure`: recommended replenishment quantity multiplied by average unit price for Stockout Risk SKUs.
- `overstock_capital_exposure`: simulated unit cost applied to inventory above 180 days of demand for Overstock Risk SKUs.

These measures are illustrative and are not real company accounting values.

In [ ]:
sku_profile["estimated_inventory_value"] = (
    sku_profile["current_inventory"] * sku_profile["unit_cost"]
).round(2)

sku_profile["stockout_revenue_exposure"] = np.where(
    sku_profile["inventory_risk"] == "Stockout Risk",
    sku_profile["recommended_replenishment_qty"] * sku_profile["avg_unit_price"],
    0,
).round(2)

sku_profile["inventory_units_above_180_days"] = np.where(
    sku_profile["inventory_risk"] == "Overstock Risk",
    (
        sku_profile["current_inventory"]
        - sku_profile["avg_daily_demand"] * 180
    ).clip(lower=0),
    0,
).round(2)

sku_profile["overstock_capital_exposure"] = (
    sku_profile["inventory_units_above_180_days"] * sku_profile["unit_cost"]
).round(2)

sku_profile[
    [
        "stock_code", "description", "sku_class", "inventory_risk",
        "estimated_inventory_value", "stockout_revenue_exposure",
        "overstock_capital_exposure",
    ]
].head(10)

## Inventory value by SKU class

This summary groups only by SKU class and reports SKU count, simulated inventory units, inventory value, stockout revenue exposure, and overstock capital exposure.

In [ ]:
inventory_value_by_sku_class = (
    sku_profile
    .groupby("sku_class", as_index=False)
    .agg(
        sku_count=("stock_code", "count"),
        total_inventory_units=("current_inventory", "sum"),
        estimated_inventory_value=("estimated_inventory_value", "sum"),
        stockout_revenue_exposure=("stockout_revenue_exposure", "sum"),
        overstock_capital_exposure=("overstock_capital_exposure", "sum"),
    )
    .sort_values("estimated_inventory_value", ascending=False)
)

financial_columns = [
    "estimated_inventory_value",
    "stockout_revenue_exposure",
    "overstock_capital_exposure",
]
for column in financial_columns:
    inventory_value_by_sku_class[column] = (
        inventory_value_by_sku_class[column].round(2)
    )

inventory_value_by_sku_class

## Inventory value by warehouse strategy

This summary connects the same simulated inventory and exposure measures to warehouse strategy.

In [ ]:
inventory_value_by_warehouse_strategy = (
    sku_profile
    .groupby("warehouse_strategy", as_index=False)
    .agg(
        sku_count=("stock_code", "count"),
        total_inventory_units=("current_inventory", "sum"),
        estimated_inventory_value=("estimated_inventory_value", "sum"),
        stockout_revenue_exposure=("stockout_revenue_exposure", "sum"),
        overstock_capital_exposure=("overstock_capital_exposure", "sum"),
    )
    .sort_values("estimated_inventory_value", ascending=False)
)

for column in financial_columns:
    inventory_value_by_warehouse_strategy[column] = (
        inventory_value_by_warehouse_strategy[column].round(2)
    )

inventory_value_by_warehouse_strategy

## Top exposure review lists

The two review lists retain notebook 05's positive-exposure filters, descending rankings, selected display fields, and 25-row limits.

In [ ]:
top_overstock_capital_exposure = (
    sku_profile[sku_profile["overstock_capital_exposure"] > 0]
    .sort_values("overstock_capital_exposure", ascending=False)[
        [
            "stock_code",
            "description",
            "sku_class",
            "warehouse_strategy",
            "current_inventory",
            "inventory_coverage_days",
            "unit_cost",
            "estimated_inventory_value",
            "overstock_capital_exposure",
        ]
    ]
    .head(25)
)

top_stockout_revenue_exposure = (
    sku_profile[sku_profile["stockout_revenue_exposure"] > 0]
    .sort_values("stockout_revenue_exposure", ascending=False)[
        [
            "stock_code",
            "description",
            "sku_class",
            "warehouse_strategy",
            "avg_monthly_units",
            "current_inventory",
            "reorder_point",
            "recommended_replenishment_qty",
            "avg_unit_price",
            "stockout_revenue_exposure",
        ]
    ]
    .head(25)
)

print("Top overstock exposure rows:", len(top_overstock_capital_exposure))
print("Top stockout exposure rows:", len(top_stockout_revenue_exposure))

## Working-capital summary

The management summary consolidates SKU counts and simulated financial exposure totals using the same definitions and two-decimal presentation as notebook 05.

In [ ]:
summary_values = {
    "total_skus": str(len(sku_profile)),
    "total_estimated_inventory_value": (
        f"{sku_profile['estimated_inventory_value'].sum():.2f}"
    ),
    "stockout_risk_skus": str(
        int((sku_profile["inventory_risk"] == "Stockout Risk").sum())
    ),
    "stockout_revenue_exposure": (
        f"{sku_profile['stockout_revenue_exposure'].sum():.2f}"
    ),
    "overstock_risk_skus": str(
        int((sku_profile["inventory_risk"] == "Overstock Risk").sum())
    ),
    "overstock_capital_exposure": (
        f"{sku_profile['overstock_capital_exposure'].sum():.2f}"
    ),
    "sku_classes": str(sku_profile["sku_class"].nunique()),
    "warehouse_strategy_count": str(
        sku_profile["warehouse_strategy"].nunique()
    ),
}

working_capital_summary = pd.DataFrame({
    "metric": list(summary_values.keys()),
    "value": list(summary_values.values()),
})

working_capital_summary

## Save standardized outputs

All five outputs are written only to `outputs_standardized/`.

In [ ]:
output_tables = {
    "working_capital_summary.csv": working_capital_summary,
    "top_overstock_capital_exposure.csv": top_overstock_capital_exposure,
    "top_stockout_revenue_exposure.csv": top_stockout_revenue_exposure,
    "inventory_value_by_sku_class.csv": inventory_value_by_sku_class,
    "inventory_value_by_warehouse_strategy.csv": (
        inventory_value_by_warehouse_strategy
    ),
}

for filename, output_table in output_tables.items():
    output_table.to_csv(outputs_dir / filename, index=False)

print("Saved output files:")
for filename in output_tables:
    print(f"- outputs_standardized/{filename}")

## Final summary

The final cell lists output files and shapes, then reports the key simulated inventory and financial exposure metrics.

All monetary values remain illustrative portfolio estimates rather than real company accounting amounts.

In [ ]:
print("===== 05b Standardized-Input Working Capital Impact Summary =====")
print("\nOutput files and shapes:")
for filename, output_table in output_tables.items():
    print(f"- outputs_standardized/{filename}: {output_table.shape}")

print("\nKey simulated financial exposure metrics:")
print(f"- Total SKUs: {len(sku_profile):,}")
print(
    "- Stockout Risk SKUs: "
    f"{int((sku_profile['inventory_risk'] == 'Stockout Risk').sum()):,}"
)
print(
    "- Overstock Risk SKUs: "
    f"{int((sku_profile['inventory_risk'] == 'Overstock Risk').sum()):,}"
)
print(
    "- Estimated inventory value: "
    f"{sku_profile['estimated_inventory_value'].sum():,.2f}"
)
print(
    "- Stockout revenue exposure: "
    f"{sku_profile['stockout_revenue_exposure'].sum():,.2f}"
)
print(
    "- Overstock capital exposure: "
    f"{sku_profile['overstock_capital_exposure'].sum():,.2f}"
)